# WAQ analysis for thick-tailed outcomes

This notebook shows the **Weighted Average Quantile (WAQ)** estimator ([arXiv:2109.02603](https://arxiv.org/abs/2109.02603)) via `analysis_type="waq"`.

**Assumptions (v1):** simple unit-level randomization (not cluster-randomized). Under a location-shift treatment effect, WAQ targets the same ATE as the mean; see [WAQ documentation](../waq.md).


In [1]:
import numpy as np
import pandas as pd

from cluster_experiments import (
    AnalysisPlan,
    HypothesisTest,
    SimpleMetric,
    Variant,
)

rng = np.random.default_rng(42)
n = 800
true_tau = 0.4
# Thick-tailed outcomes; treatment arm is control distribution + constant shift
y_control = rng.lognormal(mean=0.0, sigma=1.2, size=n)
y_treat = rng.lognormal(mean=0.0, sigma=1.2, size=n) + true_tau

df = pd.DataFrame({
    "variant": ["control"] * n + ["treatment"] * n,
    "revenue": np.concatenate([y_control, y_treat]),
})

plan_waq = AnalysisPlan(
    tests=[
        HypothesisTest(
            metric=SimpleMetric(alias="rev", name="revenue"),
            analysis_type="waq",
            analysis_config={
                "cluster_cols": [],
                "bootstrap_samples": 299,
                "random_state": 0,
                "use_numba": False,
            },
        ),
        HypothesisTest(
            metric=SimpleMetric(alias="rev_ols", name="revenue"),
            analysis_type="ols_non_clustered",
            analysis_config={},
        ),
    ],
    variants=[
        Variant(name="control", is_control=True),
        Variant(name="treatment", is_control=False),
    ],
    variant_col="variant",
    alpha=0.05,
)

results = plan_waq.analyze(df).to_dataframe()
print(results[["metric_alias", "analysis_type", "ate", "p_value"]].to_string(index=False))
print("\nDifference in arm means (DiM):", float(y_treat.mean() - y_control.mean()))


metric_alias     analysis_type      ate  p_value
         rev               waq 0.417187 0.017982
     rev_ols ols_non_clustered 0.426046 0.006518

Difference in arm means (DiM): 0.426046153527893


## Optional: faster KDE with Numba

Install `pip install 'cluster-experiments[performance]'` and set `use_numba: True` in `analysis_config` for large control samples.
